# Investigating the need of Standard Scaler


In [ ]:
import numpy as np
import pandas as pd
import pydot

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score, precision_score, balanced_accuracy_score, accuracy_score, recall_score, roc_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE, RFECV

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from catboost import CatBoostClassifier

from collections import deque
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import trange


In [ ]:
# define constants according to your dataset

DATASET = "sec_1_2_3_engineered_data.csv"
# DATASET = "sample.csv"
COLUMN_NAME_LOS = "DurationHospStayDay"
COLUMN_NAME_ADMISSION_DATE = "DateAdmission"
COLUMN_NAME_TARGET = "Outcome"

In [ ]:
df = pd.read_csv('../data/' + DATASET, low_memory=False)
df = df.drop(columns=[COLUMN_NAME_LOS])
df.drop(COLUMN_NAME_ADMISSION_DATE, axis=1, inplace=True)

In [ ]:
# Example: binary outcome
X = df.drop(COLUMN_NAME_TARGET, axis=1)
y = df[COLUMN_NAME_TARGET]

In [ ]:
label_encoder = LabelEncoder()
label_mapping = {}
for column in X.columns:
  if X[column].dtype == 'object':
    X[column] = X[column].astype(str)
    X[column] = label_encoder.fit_transform(X[column])
    label_mapping[column] = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

print(label_mapping.keys())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=123
)

## Models


In [ ]:
models = [
    {
        "model": DecisionTreeClassifier(),
        "model_name": "Decision Tree Classifier",
        "hyperparameters": {
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 5, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": [None, "sqrt", "log2"]
        }
    },
    {
        "model": RandomForestClassifier(),
        "model_name": "Random Forest Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": ExtraTreesClassifier(criterion="entropy",random_state=123),
        "model_name": "Extra Trees Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [None, 10, 20, 50],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": GradientBoostingClassifier(),
        "model_name": "Gradient Boosting Classifier",
        "hyperparameters": {
            "model__n_estimators": [100, 200, 500],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth": [3, 5, 7],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__subsample": [0.6, 0.8, 1.0],
            "model__max_features": ["sqrt", "log2", None]
        }
    },
    {
        "model": CatBoostClassifier(verbose=False),
        "model_name": "CatBoost Classifier",
        "hyperparameters": {
            "model__iterations": [200, 500, 1000],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__depth": [4, 6, 8, 10],
            "model__l2_leaf_reg": [1, 3, 5, 7]
        }
    }]

In [ ]:
scoring = {
    "f1": make_scorer(f1_score, average="binary", pos_label="Dead"),
    "roc_auc": make_scorer(roc_auc_score, needs_proba=True),
    "precision": make_scorer(precision_score, pos_label="Dead"),
    "accuracy": make_scorer(accuracy_score),
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "recall": make_scorer(recall_score, pos_label="Dead")
}

## Baseline


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

baseline_model_results = []

for model in models:
    scaled_model_result = {}
    scaled_model_result["model"] = model["model_name"]

    selected_model = model["model"]
    
    for scoring_name in scoring.keys():
        scores = cross_val_score(
            selected_model, X_train, y_train, cv=cv, scoring=scoring[scoring_name]
        )
        scaled_model_result[scoring_name] = np.mean(scores)
        
    baseline_model_results.append(scaled_model_result)

In [ ]:
baseline_model_results_df = pd.DataFrame(baseline_model_results)
baseline_model_results_df.round(4)

## With Standard Scaler

In [ ]:
from sklearn.discriminant_analysis import StandardScaler


scaled_model_results = []

for model in models:
    scaled_model_result = {}
    scaled_model_result["model"] = model["model_name"]

    selected_model = model["model"]
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model["model"])
    ])
    
    for scoring_name in scoring.keys():
        scores = cross_val_score(
            selected_model, X_train, y_train, cv=cv, scoring=scoring[scoring_name]
        )
        scaled_model_result[scoring_name] = np.mean(scores)
        
    scaled_model_results.append(scaled_model_result)

In [ ]:
scaled_model_results_df = pd.DataFrame(scaled_model_results)
scaled_model_results_df.round(4)

## Comparing Results

In [ ]:
# Compare average of all scores

for score in scoring.keys():
    print("==========================")
    print(f"comparing: {score}")
    print(f"before scaling: {baseline_model_results_df[score].mean()}")
    print(f"after scaling: {scaled_model_results_df[score].mean()}")
    print(f"Impact of scaling on {score}: {scaled_model_results_df[score].mean() - baseline_model_results_df[score].mean()}")

In [ ]:
# paired_cv_stat_tests.py
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, balanced_accuracy_score
)
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

# ---------------------------
# DeLong implementation (fast, public-domain style)
# ---------------------------
def _compute_midrank(x):
    """Compute midranks for an array x (ascending)."""
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j + 1 < N and Z[j + 1] == Z[i]:
            j += 1
        # ranks i..j inclusive
        rank = 0.5 * (i + j) + 1  # +1 for 1-based ranks (not necessary for variance but kept)
        T[i:j+1] = rank
        i = j + 1
    # put back to original order
    midranks = np.empty(N, dtype=float)
    midranks[J] = T
    return midranks

def _fast_delong(predictions_sorted_transposed, label_1_count):
    """
    Core fast DeLong as used in multiple public implementations.
    predictions_sorted_transposed: shape (n_models, n_samples) sorted by prediction descending
    label_1_count: number of positive samples (n_pos)
    Returns: (aucs, covariance matrix)
    """
    m = predictions_sorted_transposed.shape[0]
    n = predictions_sorted_transposed.shape[1]

    # Number of positives
    k = label_1_count
    # Number of negatives
    l = n - k

    # Split predictions into pos and neg
    pos_preds = predictions_sorted_transposed[:, :k]
    neg_preds = predictions_sorted_transposed[:, k:]

    # compute midranks for pos and neg separately and combined
    # combine per-model
    def compute_v(x):
        # x: 1d array shape (n_samples,)
        return _compute_midrank(x)

    # For each model compute:
    # V10 = (R_pos - (k+1)/2) / l
    # V01 = 1 - (R_neg - (k+1)/2) / k  (or similar forms)
    tx = np.empty((m, k), dtype=float)
    ty = np.empty((m, l), dtype=float)

    for r in range(m):
        all_scores = np.concatenate([pos_preds[r], neg_preds[r]])
        # we need midranks on the combined array sorted descending because we passed
        # a pre-sorted array; but many implementations assume ascending, so adapt:
        # We'll compute midranks on combined ascending
        midranks_all = _compute_midrank(all_scores)
        tx[r, :] = midranks_all[:k]
        ty[r, :] = midranks_all[k:]

    aucs = (np.sum(tx, axis=1) - (k * (k + 1) / 2.0)) / (k * l)

    # V-statistics: variance components
    v01 = (tx - np.mean(tx, axis=1, keepdims=True)) / l
    v10 = (ty - np.mean(ty, axis=1, keepdims=True)) / k

    S = np.cov(np.concatenate([v01, v10], axis=1))
    # The covariance matrix for AUCs is S / n_samples? Many references build it differently.
    # Here we follow the common formula: covariance = S / n_samples
    return aucs, S

def delong_roc_test(y_true, y_scores_A, y_scores_B):
    """
    Perform DeLong test for two correlated ROC AUCs.
    Returns: z_statistic, p_value, auc_A, auc_B
    This implementation follows the public fast DeLong algorithm.
    """
    y_true = np.asarray(y_true)
    assert set(np.unique(y_true)).issubset({0, 1}), "y_true must be 0/1"

    # stack predictions for two models
    preds = np.vstack([y_scores_A, y_scores_B])
    # sort by label (positives first) required by our _fast_delong function:
    # many implementations reorder so positives appear before negatives:
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]
    order = np.concatenate([pos_idx, neg_idx])
    # reorder predictions accordingly
    preds_sorted = preds[:, order]

    n_pos = len(pos_idx)
    aucs, cov = _fast_delong(preds_sorted, n_pos)
    # covariance of difference:
    var_diff = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var_diff <= 0:
        # fallback small positive to avoid divide-by-zero
        var_diff = 1e-10
    z = (aucs[0] - aucs[1]) / np.sqrt(var_diff)
    # two-sided p-value from z
    from scipy.stats import norm
    p = 2 * (1 - norm.cdf(abs(z)))
    return float(z), float(p), float(aucs[0]), float(aucs[1])

# ---------------------------
# Main repeated-CV evaluation + stats
# ---------------------------
def evaluate_models_with_cv_and_pvals(models, X, y,
                                      pos_label='Dead',
                                      n_splits=5, n_repeats=10,
                                      random_state=42):
    """
    models: list of dicts: [{'model_name': str, 'model': sklearn estimator}, ...]
    X: features (numpy/pandas)
    y: labels (array-like of strings), positive class indicated by `pos_label`
    Returns: DataFrame with test statistics and p-values (FDR corrected)
    """
    # encode labels to 0/1 consistently
    le = LabelEncoder()
    y_bin = le.fit_transform(y)
    assert pos_label in le.classes_, f"pos_label {pos_label} not present in labels"

    # which code value corresponds to pos_label
    pos_code = int(np.where(le.classes_ == pos_label)[0][0])

    cv = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                 random_state=random_state)

    # storage: per model, per fold metric arrays
    per_model_metrics = {}
    n_folds = n_splits * n_repeats

    metric_fns = {
        'accuracy': lambda yt, yp, ps: accuracy_score(yt, yp),
        'roc_auc': lambda yt, yp, ps: roc_auc_score(yt, ps),
        'f1': lambda yt, yp, ps: f1_score(yt, yp, zero_division=0),
        'precision': lambda yt, yp, ps: precision_score(yt, yp, zero_division=0),
        'recall': lambda yt, yp, ps: recall_score(yt, yp, zero_division=0),
        'balanced_accuracy': lambda yt, yp, ps: balanced_accuracy_score(yt, yp)
    }

    for m in models:
        name = m['model_name']
        per_model_metrics[name] = {metric: [] for metric in metric_fns.keys()}
        per_model_metrics[name]['raw_auc_pairs'] = []  # store predicted scores pair for optional DeLong per fold

    # iterate CV
    fold_i = 0
    for train_idx, test_idx in cv.split(X, y_bin):
        fold_i += 1
        X_tr, X_te = (X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx],
                      X.iloc[test_idx] if hasattr(X, "iloc") else X[test_idx])
        y_tr, y_te = y_bin[train_idx], y_bin[test_idx]

        for m in models:
            name = m['model_name']
            base_clf = m['model'].__class__(**m['model'].get_params())
            base_clf.fit(X_tr, y_tr)
            yhat_base = base_clf.predict(X_te)
            prob_base = (base_clf.predict_proba(X_te)[:, 1]
                         if hasattr(base_clf, "predict_proba") else base_clf.decision_function(X_te))

            # scaled pipeline
            scaled_pipeline = Pipeline([('scaler', StandardScaler()),
                                        ('model', m['model'].__class__(**m['model'].get_params()))])
            scaled_pipeline.fit(X_tr, y_tr)
            yhat_scaled = scaled_pipeline.predict(X_te)
            prob_scaled = (scaled_pipeline.predict_proba(X_te)[:, 1]
                           if hasattr(scaled_pipeline, "predict_proba") else scaled_pipeline.decision_function(X_te))

            # ensure integer 0/1 predictions (in case of string labels)
            yhat_base = le.inverse_transform(yhat_base) if isinstance(yhat_base[0], (str,)) else yhat_base
            # but we encoded y to 0/1 and trained on 0/1 so predictions should be 0/1 already

            # compute metrics for this fold
            for metric, fn in metric_fns.items():
                if metric == 'roc_auc':
                    s_base = fn(y_te, yhat_base, prob_base)
                    s_scaled = fn(y_te, yhat_scaled, prob_scaled)
                else:
                    s_base = fn(y_te, yhat_base, prob_base)
                    s_scaled = fn(y_te, yhat_scaled, prob_scaled)
                per_model_metrics[name][metric].append((s_base, s_scaled))

            # store raw score arrays for potential per-fold DeLong
            per_model_metrics[name]['raw_auc_pairs'].append((y_te.copy(), prob_base.copy(), prob_scaled.copy()))

    # Build results and run Wilcoxon paired test across folds
    rows = []
    for name, metrics_dict in per_model_metrics.items():
        for metric in metric_fns.keys():
            # extract paired arrays
            paired = np.array(metrics_dict[metric])  # shape (n_folds, 2)
            scores_base = paired[:, 0]
            scores_scaled = paired[:, 1]
            # compute median/mean difference
            mean_base = np.mean(scores_base)
            mean_scaled = np.mean(scores_scaled)
            
            # If completely identical, skip statistical test
            differences = scores_base - scores_scaled
            if np.all(differences == 0):
                rows.append({
                    'model': name,
                    'metric': metric,
                    'n_folds': len(scores_base),
                    'mean_base': mean_base,
                    'mean_scaled': mean_scaled,
                    'statistic': 0.0,
                    'p_value_raw': 1.0,
                    'note': 'identical_across_folds'
                })
                continue
            
            try:
                # choose Wilcoxon (nonparametric paired)
                stat, p = wilcoxon(scores_base, scores_scaled, alternative='two-sided')
            except ValueError:
                # if all differences are zero or n too small, fallback to t-test
                from scipy.stats import ttest_rel
                stat, p = ttest_rel(scores_base, scores_scaled)
            rows.append({
                'model': name,
                'metric': metric,
                'n_folds': len(scores_base),
                'mean_base': mean_base,
                'mean_scaled': mean_scaled,
                'statistic': float(stat),
                'p_value_raw': float(p)
            })

        # Optionally compute DeLong on aggregated holdout across all folds or per-fold.
        # Here we compute DeLong on the concatenation of all test sets (paired scores on same samples no longer guaranteed).
        # Better: compute DeLong per fold and then wilcoxon on per-fold z-statistics — left as optional.
    results_df = pd.DataFrame(rows)

    # multiple testing correction (all tests together)
    rej, pvals_corr, _, _ = multipletests(results_df['p_value_raw'].values, method='fdr_bh')
    results_df['p_value_fdr'] = pvals_corr
    results_df['significant_fdr'] = rej

    return results_df, per_model_metrics

# ---------------------------
# Example usage (adapt to your environment)
# ---------------------------
# Suppose you have:
# models = [
#   {'model_name': 'logreg', 'model': LogisticRegression(max_iter=1000)},
#   {'model_name': 'rf', 'model': RandomForestClassifier(n_estimators=200)}
#   ... up to 5 models
# ]
# X, y are your data (pandas DataFrame / Series). y contains string labels including "Dead".
#
results_df, per_model_metrics = evaluate_models_with_cv_and_pvals(models, X, y,
                                                                  pos_label='Dead',
                                                                  n_splits=5, n_repeats=10)
display(results_df.sort_values(['model','metric']))


In [ ]:
results_df.to_csv("scaling_statistical_testing_results.csv")

In [ ]:
import numpy as np

metric_fns = {
    'accuracy': lambda yt, yp, ps: accuracy_score(yt, yp),
    'roc_auc': lambda yt, yp, ps: roc_auc_score(yt, ps),
    'f1': lambda yt, yp, ps: f1_score(yt, yp, zero_division=0),
    'precision': lambda yt, yp, ps: precision_score(yt, yp, zero_division=0),
    'recall': lambda yt, yp, ps: recall_score(yt, yp, zero_division=0),
    'balanced_accuracy': lambda yt, yp, ps: balanced_accuracy_score(yt, yp)
}

for name, metrics_dict in per_model_metrics.items():
    for metric in metric_fns.keys():
        arr = np.array(metrics_dict[metric])  # shape (n_folds, 2)
        base = np.array(arr)[:,0]
        scaled = np.array(arr)[:,1]

        if len(set(base)) == 1:
            print(name, metric, "base is constant:", base[0])
        if len(set(scaled)) == 1:
            print(name, metric, "scaled is constant:", scaled[0])
        if np.all(base == scaled):
            print(name, metric, "base == scaled (identical arrays)")
        if np.isnan(base).any() or np.isnan(scaled).any():
            print(model, metric, "contains NaN")


## Viz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_difference(per_model_metrics, model_name, metric):
    paired = np.array(per_model_metrics[model_name][metric])   # shape (n_folds, 2)
    base = paired[:, 0]
    scaled = paired[:, 1]
    diff = scaled - base

    plt.figure(figsize=(10, 4))
    plt.plot(diff, marker='o', linestyle='-', alpha=0.8)
    plt.axhline(0, color='black', linestyle='--', linewidth=1)

    plt.title(f"Per-fold difference (scaled - base)\n{model_name} — {metric}")
    plt.xlabel("Fold index")
    plt.ylabel("Difference")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_difference(per_model_metrics, "CatBoost Classifier", "roc_auc")

In [ ]:
plot_difference(per_model_metrics, "Extra Trees Classifier", "roc_auc")

In [ ]:
def plot_boxplot(per_model_metrics, model_name, metric):
    paired = np.array(per_model_metrics[model_name][metric])
    base = paired[:, 0]
    scaled = paired[:, 1]

    plt.figure(figsize=(6, 5))
    plt.boxplot([base, scaled],
                labels=['Base', 'Scaled'],
                showmeans=True)

    plt.title(f"{model_name}\n{metric} distribution across folds")
    plt.ylabel(metric)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_boxplot(per_model_metrics, "Decision Tree Classifier", "roc_auc")

In [ ]:
def plot_all_models_difference(per_model_metrics, metric, models):
    n = len(models)
    fig, axes = plt.subplots(n, 1, figsize=(10, 3 * n), sharex=True)

    for i, model_name in enumerate(models):
        paired = np.array(per_model_metrics[model_name][metric])
        base = paired[:, 0]
        scaled = paired[:, 1]
        diff = scaled - base

        axes[i].plot(diff, marker='o', linestyle='-')
        axes[i].axhline(0, color='black', linestyle='--', linewidth=1)
        axes[i].set_title(f"{model_name}")
        axes[i].set_ylabel("Δ Score")
        axes[i].grid(True, linestyle='--', alpha=0.4)

    axes[-1].set_xlabel("Fold index")
    fig.suptitle(f"Scaled - Base Differences for {metric}", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


In [ ]:
models_to_plot = [
    "Decision Tree Classifier",
    "Random Forest Classifier",
    "Extra Trees Classifier",
    "Gradient Boosting Classifier",
    "CatBoost Classifier"
]
plot_all_models_difference(per_model_metrics, "roc_auc", models_to_plot)